# Robust nodal RSAM summaries for selected time windows

This notebook replaces the original stage-70 summary notebook.

It reads the position-coded RSAM archive directly:

```text
/Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes
```

The station code is interpreted as position in centimetres, so no external
geometry or serial-number table is required. RSAM files ending in both
`_59s.csv` and `_60s.csv` are read and combined.

The output is a tidy CSV containing robust median amplitudes for:

- broadband one-minute median amplitude; and
- each stored bandpass metric.

The summary is intended as an intermediate, reproducible table for the spatial
profile notebook. It does not create figures.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from obspy import UTCDateTime

from flovopy.processing.sam import RSAM

## Configuration

In [2]:
SAM_DIR = Path(
    "/Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes"
)
OUT_DIR = SAM_DIR / "noise_profiles"
OUT_DIR.mkdir(parents=True, exist_ok=True)

NETWORKS = ["T1", "T3"]
CHANNELS = ["DPE", "DPN", "DPZ"]
COMPONENT_LABELS = {
    "DPE": "E",
    "DPN": "N",
    "DPZ": "Z",
}

METRICS = [
    "median",
    "B4_8",
    "B8_16",
    "B16_32",
    "B32_64",
    "B64_128",
    "B128_240",
]

SAMPLING_INTERVALS_S = [59, 60]
RSAM_EXTENSION = "csv"

# Convert stored voltage amplitudes to velocity in m/s.
SENSITIVITY = 2000.0  # V / (m/s)

POSITION_TOLERANCE_M = 0.25

WINDOWS = [
    {
        "period": "quiet_2026-05-17_0400_0800",
        "group": "quiet",
        "label": "17 May 04:00–08:00 UTC",
        "start": UTCDateTime("2026-05-17T04:00:00"),
        "end": UTCDateTime("2026-05-17T08:00:00"),
    },
    {
        "period": "quiet_2026-05-18_0400_0800",
        "group": "quiet",
        "label": "18 May 04:00–08:00 UTC",
        "start": UTCDateTime("2026-05-18T04:00:00"),
        "end": UTCDateTime("2026-05-18T08:00:00"),
    },
    {
        "period": "quiet_2026-05-19_0400_0800",
        "group": "quiet",
        "label": "19 May 04:00–08:00 UTC",
        "start": UTCDateTime("2026-05-19T04:00:00"),
        "end": UTCDateTime("2026-05-19T08:00:00"),
    },
    {
        "period": "elevated_2026-05-18_1100_1300",
        "group": "elevated_pre_survey",
        "label": "18 May 11:00–13:00 UTC",
        "start": UTCDateTime("2026-05-18T11:00:00"),
        "end": UTCDateTime("2026-05-18T13:00:00"),
    },
    {
        "period": "survey_T1_1m",
        "group": "active_survey",
        "label": "T1 1 m, 18 May 16:03–18:39 UTC",
        "start": UTCDateTime("2026-05-18T16:03:00"),
        "end": UTCDateTime("2026-05-18T18:39:00"),
    },
    {
        "period": "survey_T1_2m",
        "group": "active_survey",
        "label": "T1 2 m, 18 May 20:18–23:13 UTC",
        "start": UTCDateTime("2026-05-18T20:18:00"),
        "end": UTCDateTime("2026-05-18T23:13:00"),
    },
    {
        "period": "survey_nodal_only",
        "group": "active_survey",
        "label": "Nodal only, 19 May 12:59–16:02 UTC",
        "start": UTCDateTime("2026-05-19T12:59:00"),
        "end": UTCDateTime("2026-05-19T16:02:00"),
    },
]

START = min(window["start"] for window in WINDOWS)
END = max(window["end"] for window in WINDOWS)

print(f"RSAM root: {SAM_DIR}")
print(f"Output:    {OUT_DIR}")
print(f"Read:      {START} to {END}")

RSAM root: /Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes
Output:    /Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes/noise_profiles
Read:      2026-05-17T04:00:00.000000Z to 2026-05-19T16:02:00.000000Z


## Helper functions

In [3]:
def parse_seed_id(seed_id: str) -> tuple[str, str, str, str]:
    parts = seed_id.split(".")
    if len(parts) != 4:
        raise ValueError(
            f"Expected NET.STA.LOC.CHA, got {seed_id!r}"
        )
    return tuple(parts)


def station_code_to_position_m(station: str) -> float:
    text = str(station).strip()

    if text.endswith(".0"):
        text = text[:-2]

    if not text.isdigit():
        raise ValueError(
            f"Station code {station!r} is not an integer centimetre position."
        )

    return int(text) / 100.0


def cluster_position_values(
    values: pd.Series,
    tolerance_m: float,
) -> tuple[pd.Series, pd.DataFrame]:
    unique = np.sort(
        values.dropna().unique().astype(float)
    )

    if len(unique) == 0:
        return values.copy(), pd.DataFrame()

    clusters = []
    current = [float(unique[0])]

    for value in unique[1:]:
        value = float(value)

        if value - current[0] <= tolerance_m + 1e-12:
            current.append(value)
        else:
            clusters.append(current)
            current = [value]

    clusters.append(current)

    lookup = {}
    rows = []

    for cluster_id, members in enumerate(clusters, start=1):
        merged = float(np.mean(members))

        for member in members:
            lookup[member] = merged

        rows.append(
            {
                "cluster": cluster_id,
                "position_m": merged,
                "minimum_original_m": min(members),
                "maximum_original_m": max(members),
                "span_m": max(members) - min(members),
                "original_positions_m": ", ".join(
                    f"{member:.2f}" for member in members
                ),
            }
        )

    clustered = values.map(
        lambda value: (
            lookup[float(value)]
            if pd.notna(value)
            else np.nan
        )
    )

    return clustered, pd.DataFrame(rows)


def read_rsam_multiple_intervals(
    *,
    network: str,
    start: UTCDateTime,
    end: UTCDateTime,
) -> RSAM:
    combined = None

    for interval_s in SAMPLING_INTERVALS_S:
        partial = RSAM.read(
            start,
            end,
            SAM_DIR=str(SAM_DIR),
            network=network,
            sampling_interval=interval_s,
            ext=RSAM_EXTENSION,
            verbose=False,
        )

        print(
            f"{network}: {interval_s}s files -> "
            f"{len(partial.dataframes)} dataframes"
        )

        if combined is None:
            combined = partial
        else:
            duplicate_ids = (
                set(combined.dataframes)
                & set(partial.dataframes)
            )

            if duplicate_ids:
                raise ValueError(
                    "Duplicate IDs across nominal sampling intervals: "
                    f"{sorted(duplicate_ids)[:10]}"
                )

            combined.dataframes.update(
                partial.dataframes
            )

    return combined

## Read the RSAM archive

In [4]:
rsam_by_network = {
    network: read_rsam_multiple_intervals(
        network=network,
        start=START,
        end=END,
    )
    for network in NETWORKS
}

for network, rsam in rsam_by_network.items():
    print(
        f"{network}: {len(rsam.dataframes)} combined dataframes"
    )

creating blank SAM object
T1: 59s files -> 0 dataframes
Dataframe with 780 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 780 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 780 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 70 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 70 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 70 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 780 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 780 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 780 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 69 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 69 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 69 rows is already on a regular 60 s grid b

## Summarize each position and time window

The stored one-minute values are divided by `SENSITIVITY` before any temporal
summary is calculated. For every station/window/metric, the notebook retains
the median of the one-minute values.

In [5]:
rows = []
skipped = []

for network, rsam in rsam_by_network.items():
    for seed_id, dataframe in rsam.dataframes.items():
        try:
            net, station, location, channel = parse_seed_id(
                seed_id
            )

            if net != network or channel not in CHANNELS:
                continue

            position_original_m = (
                station_code_to_position_m(station)
            )

            if isinstance(dataframe.index, pd.DatetimeIndex):
                times = pd.to_datetime(
                    dataframe.index,
                    utc=True,
                )
            elif "time" in dataframe.columns:
                times = pd.to_datetime(
                    dataframe["time"],
                    unit="s",
                    utc=True,
                    errors="coerce",
                )
            else:
                raise KeyError("No usable time index or time column.")

        except Exception as exc:
            skipped.append((seed_id, str(exc)))
            continue

        for window in WINDOWS:
            start = pd.Timestamp(
                window["start"].datetime,
                tz="UTC",
            )
            end = pd.Timestamp(
                window["end"].datetime,
                tz="UTC",
            )
            mask = (times >= start) & (times < end)

            for metric in METRICS:
                if metric not in dataframe.columns:
                    continue

                values = pd.to_numeric(
                    dataframe.loc[mask, metric],
                    errors="coerce",
                ) / SENSITIVITY

                values = values[
                    np.isfinite(values)
                    & (values > 0)
                ]

                if values.empty:
                    continue

                rows.append(
                    {
                        "network": net,
                        "station": station,
                        "location": location,
                        "channel": channel,
                        "component": COMPONENT_LABELS[channel],
                        "position_original_m": position_original_m,
                        "period": window["period"],
                        "group": window["group"],
                        "period_label": window["label"],
                        "start_utc": start,
                        "end_utc": end,
                        "metric": metric,
                        "n_minutes": len(values),
                        "median_velocity_mps": values.median(),
                    }
                )

summary = pd.DataFrame(rows)

if summary.empty:
    raise RuntimeError("No RSAM window summaries were produced.")

summary["position_m"], position_clusters = (
    cluster_position_values(
        summary["position_original_m"],
        POSITION_TOLERANCE_M,
    )
)

summary = summary.sort_values(
    [
        "network",
        "component",
        "position_m",
        "metric",
        "start_utc",
    ]
).reset_index(drop=True)

print(f"Summary rows: {len(summary):,}")
print(f"Skipped IDs:  {len(skipped)}")
display(summary.head())

Summary rows: 6,489
Skipped IDs:  0


,network,station,location,channel,component,position_original_m,period,group,period_label,start_utc,end_utc,metric,n_minutes,median_velocity_mps,position_m
0,T1,02800,N1,DPE,E,28.0,quiet_2026-05-17_0400_0800,quiet,17 May 04:00–08:00 UTC,2026-05-17 04:00:00+00:00,2026-05-17 08:00:00+00:00,B128_240,240,4.229233e-07,28.0
1,T1,02800,N3,DPE,E,28.0,survey_nodal_only,active_survey,"Nodal only, 19 May 12:59–16:02 UTC",2026-05-19 12:59:00+00:00,2026-05-19 16:02:00+00:00,B128_240,70,4.524244e-07,28.0
2,T1,02800,N1,DPE,E,28.0,quiet_2026-05-17_0400_0800,quiet,17 May 04:00–08:00 UTC,2026-05-17 04:00:00+00:00,2026-05-17 08:00:00+00:00,B16_32,240,5.359352e-07,28.0
3,T1,02800,N3,DPE,E,28.0,survey_nodal_only,active_survey,"Nodal only, 19 May 12:59–16:02 UTC",2026-05-19 12:59:00+00:00,2026-05-19 16:02:00+00:00,B16_32,70,2.140494e-06,28.0
4,T1,02800,N1,DPE,E,28.0,quiet_2026-05-17_0400_0800,quiet,17 May 04:00–08:00 UTC,2026-05-17 04:00:00+00:00,2026-05-17 08:00:00+00:00,B32_64,240,2.282904e-07,28.0


## Save and inspect

In [6]:
summary_file = (
    OUT_DIR
    / "nodal_noise_window_summary_position_codes.csv"
)
clusters_file = (
    OUT_DIR
    / "nodal_noise_position_clusters.csv"
)

summary.to_csv(summary_file, index=False)
position_clusters.to_csv(clusters_file, index=False)

print(f"Wrote {summary_file}")
print(f"Wrote {clusters_file}")

display(
    summary
    .groupby(
        [
            "network",
            "location",
            "component",
            "metric",
            "period",
        ]
    )
    .agg(
        positions=("position_m", "nunique"),
        median_minutes=("n_minutes", "median"),
    )
    .reset_index()
)

Wrote /Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes/noise_profiles/nodal_noise_window_summary_position_codes.csv
Wrote /Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes/noise_profiles/nodal_noise_position_clusters.csv


,network,location,component,metric,period,positions,median_minutes
0,T1,N1,E,B128_240,quiet_2026-05-17_0400_0800,35,240.0
1,T1,N1,E,B16_32,quiet_2026-05-17_0400_0800,35,240.0
2,T1,N1,E,B32_64,quiet_2026-05-17_0400_0800,35,240.0
3,T1,N1,E,B4_8,quiet_2026-05-17_0400_0800,35,240.0
4,T1,N1,E,B64_128,quiet_2026-05-17_0400_0800,35,240.0
...,...,...,...,...,...,...,...
184,T3,N4,Z,B32_64,survey_nodal_only,35,39.0
185,T3,N4,Z,B4_8,survey_nodal_only,35,39.0
186,T3,N4,Z,B64_128,survey_nodal_only,35,39.0
187,T3,N4,Z,B8_16,survey_nodal_only,35,39.0
